Week 16 · Day 2 — Ingestion Pipeline (clean → chunk → embed → store)
Why this matters (2–3 lines)

Your bot is only as good as its context. Today you’ll build the ingestion pipeline that turns raw docs into searchable vectors, so tomorrow’s backend can retrieve the right snippets fast.

Theory Essentials (≤6 bullets)

Normalize text (lowercase, strip weird whitespace, drop boilerplate).

Chunking beats whole-doc embeddings (smaller, focused, retrievable).

Overlap (e.g., 50–100 chars) keeps context continuity across chunks.

Embeddings: start simple (TF-IDF) → later swap to SentenceTransformers.

Vector store: index vectors + keep metadata (doc_id, chunk_id, text).

Reproducibility: fixed params (chunk size, overlap), saved artifacts.

In [5]:
# Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
np.random.seed(42)
plt.rcParams["figure.figsize"] = (6,4)
plt.rcParams["axes.grid"] = True

# ---- Config (adjust paths as needed) ---------------------------------------
DATA_DIR       = Path("data/sample_docs")     # put your .txt/.md/.pdf (txt-extracted) here
VECTOR_DIR     = Path("vector_store")         # where we store index + metadata
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE     = 750        # chars per chunk (≈ 120–150 tokens)
CHUNK_OVERLAP  = 150        # chars of overlap between chunks
USE_TFIDF      = True       # set False later if you swap to sentence-transformers
N_NEIGHBORS    = 10         # retrieval default for quick tests

# ---- Utilities --------------------------------------------------------------
import re, json, joblib
from typing import List, Dict

def read_text_files(root: Path) -> Dict[str, str]:
    texts = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".txt", ".md"}:
            texts[p.stem] = p.read_text(encoding="utf-8", errors="ignore")
    return texts

def basic_clean(s: str) -> str:
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def chunk_text(text: str, size: int = 750, overlap: int = 150) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + size)
        chunk = text[start:end]
        chunks.append(chunk)
        if end == len(text): break
        start = end - overlap  # step with overlap
    return chunks

# ---- Embeddings backend (TF-IDF baseline; pluggable) ------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

class TFIDFIndexer:
    def __init__(self):
        self.vectorizer = None
        self.nn = None
        self.matrix = None

    def fit(self, texts: List[str]):
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_df=0.9)
        self.matrix = self.vectorizer.fit_transform(texts)
        # cosine distance = 1 - cosine_similarity; NearestNeighbors with metric="cosine"
        self.nn = NearestNeighbors(metric="cosine", n_neighbors=N_NEIGHBORS).fit(self.matrix)

    def encode(self, texts: List[str]):
        return self.vectorizer.transform(texts)

    def save(self, path: Path):
        joblib.dump({
            "vectorizer": self.vectorizer,
            "matrix": self.matrix,
            "nn": self.nn
        }, path)

    @staticmethod
    def load(path: Path):
        obj = joblib.load(path)
        idx = TFIDFIndexer()
        idx.vectorizer = obj["vectorizer"]
        idx.matrix = obj["matrix"]
        idx.nn = obj["nn"]
        return idx

# ---- Pipeline: ingest -> clean -> chunk -> embed -> store -------------------
def build_index(data_dir: Path, vector_dir: Path):
    # 0) Sample docs fallback (so the cell runs even if folder is empty)
    vector_dir.mkdir(parents=True, exist_ok=True)
    data_dir.mkdir(parents=True, exist_ok=True)
    if not any(data_dir.glob("*")):
        (data_dir/"intro.md").write_text(
            "# Course Notes\nRAG combines retrieval with generation.\n" 
            "We split documents into chunks and embed them to enable similarity search.\n"
            "High-quality chunking improves answer grounding.", encoding="utf-8"
        )
        (data_dir/"faq.txt").write_text(
            "Q: What is chunk overlap?\nA: The number of characters repeated between adjacent chunks.\n"
            "Q: Why citations?\nA: To show sources and reduce hallucinations.", encoding="utf-8"
        )

    # 1) Load & clean
    raw_docs = read_text_files(data_dir)
    cleaned_docs = {k: basic_clean(v.lower()) for k, v in raw_docs.items()}

    # 2) Chunk
    records = []  # list of dicts with {doc_id, chunk_id, text}
    for doc_id, text in cleaned_docs.items():
        chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        for i, ch in enumerate(chunks):
            records.append({"doc_id": doc_id, "chunk_id": i, "text": ch})
    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("No chunks created. Ensure there are .txt/.md files in data_dir.")

    # 3) Embed (TF-IDF baseline)
    indexer = TFIDFIndexer()
    indexer.fit(df["text"].tolist())

    # 4) Persist artifacts
    #   - vectors + index
    index_path = vector_dir / "tfidf_index.joblib"
    indexer.save(index_path)
    #   - metadata (so we can resolve chunk -> original text/source)
    meta_path = vector_dir / "chunks.parquet"
    df.to_parquet(meta_path, index=False)

    # 5) Quick smoke test: retrieve for a test query
    query = "why use chunk overlap?"
    q_vec = indexer.encode([query])

    k = min(3, len(df))  # <-- cap neighbors to available chunks
    distances, indices = indexer.nn.kneighbors(q_vec, n_neighbors=k)

    preview = df.iloc[indices[0]][["doc_id","chunk_id","text"]].copy()
    print("Built index ✅")
    print(f"Docs loaded: {len(cleaned_docs)} | Chunks: {len(df)}")
    print("\nTop-{} retrieval preview for query:".format(k), query)
    display(preview)


    return {
        "index_path": str(index_path),
        "meta_path": str(meta_path),
        "n_docs": len(cleaned_docs),
        "n_chunks": len(df)
    }

artifacts = build_index(DATA_DIR, VECTOR_DIR)

# ---- (Optional) Helper to load later in backend -----------------------------
def load_vector_store(vector_dir: Path):
    idx = TFIDFIndexer.load(vector_dir / "tfidf_index.joblib")
    meta = pd.read_parquet(vector_dir / "chunks.parquet")
    return idx, meta

# quick check
_ = load_vector_store(VECTOR_DIR)


Built index ✅
Docs loaded: 2 | Chunks: 2

Top-2 retrieval preview for query: why use chunk overlap?


,doc_id,chunk_id,text
0,faq,0,q: what is chunk overlap? a: the number of cha...
1,intro,0,# course notes rag combines retrieval with gen...


1) Core (10–15 min)
Task: Change CHUNK_SIZE to 500 and CHUNK_OVERLAP to 100, rebuild, and report how many chunks you get.

In [8]:
# after editing constants, re-run build
CHUNK_SIZE, CHUNK_OVERLAP = 500, 100
artifacts = build_index(DATA_DIR, VECTOR_DIR)
print("Chunks:", artifacts["n_chunks"])


Built index ✅
Docs loaded: 2 | Chunks: 2

Top-2 retrieval preview for query: why use chunk overlap?


,doc_id,chunk_id,text
0,faq,0,q: what is chunk overlap? a: the number of cha...
1,intro,0,# course notes rag combines retrieval with gen...


Chunks: 2


2) Practice (10–15 min)
Task: Swap the TfidfVectorizer to use unigrams only and compare top-3 retrieval results for the same query.

In [10]:
# set unigram only
from sklearn.feature_extraction.text import TfidfVectorizer
class TFIDFIndexerUni(TFIDFIndexer):
    def fit(self, texts):
        self.vectorizer = TfidfVectorizer(ngram_range=(1,1), min_df=1, max_df=0.9)
        self.matrix = self.vectorizer.fit_transform(texts)
        self.nn = NearestNeighbors(metric="cosine", n_neighbors=N_NEIGHBORS).fit(self.matrix)

# quick benchmark
df = pd.read_parquet(VECTOR_DIR/"chunks.parquet")
idx_uni = TFIDFIndexerUni(); idx_uni.fit(df["text"].tolist())
q = "why use chunk overlap?"
d, ind = idx_uni.nn.kneighbors(idx_uni.encode([q]), n_neighbors=2)
display(df.iloc[ind[0]][["doc_id","chunk_id","text"]])


,doc_id,chunk_id,text
0,faq,0,q: what is chunk overlap? a: the number of cha...
1,intro,0,# course notes rag combines retrieval with gen...


3) Stretch (optional, 10–15 min)
Task: Add a simple stopword removal step before vectorization (use TfidfVectorizer(stop_words='english')) and see if the retrieved chunks look more on-topic.

In [12]:
class TFIDFIndexerStop(TFIDFIndexer):
    def fit(self, texts):
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_df=0.9)
        self.matrix = self.vectorizer.fit_transform(texts)
        self.nn = NearestNeighbors(metric="cosine", n_neighbors=N_NEIGHBORS).fit(self.matrix)

df = pd.read_parquet(VECTOR_DIR/"chunks.parquet")
idx_sw = TFIDFIndexerStop(); idx_sw.fit(df["text"].tolist())
q = "citations and hallucinations"
d, ind = idx_sw.nn.kneighbors(idx_sw.encode([q]), n_neighbors=2)
display(df.iloc[ind[0]][["doc_id","chunk_id","text"]])


,doc_id,chunk_id,text
0,faq,0,q: what is chunk overlap? a: the number of cha...
1,intro,0,# course notes rag combines retrieval with gen...


Mini-Challenge (≤40 min)

Build app/ingestion.py that exposes a CLI:

python -m app.ingestion --data_dir data/sample_docs --out_dir vector_store --chunk_size 750 --overlap 150


Acceptance Criteria

✅ Reads .txt/.md files, cleans + chunks with overlap.

✅ Builds TF-IDF index and saves tfidf_index.joblib + chunks.parquet.

✅ Prints a short retrieval preview for a test query.

✅ Deterministic (same params → same artifacts).

Notes / Key Takeaways (5–7 bullets)

Clean, chunked text with overlap is the backbone of good retrieval.

Start with TF-IDF for speed and determinism; swap to dense embeddings later.

Always save both vectors/index and chunk metadata (to show sources).

Keep ingestion idempotent: same inputs + params → same artifacts.

Tomorrow’s backend will load tfidf_index.joblib + chunks.parquet for retrieval.

Reflection (2 prompts)

Where might your current documents need cleaning rules (headers, footers, page numbers)?

What chunk size/overlap tradeoff makes sense for your domain (short FAQs vs long PDFs)?

1) Where might your current documents need cleaning rules?

If you’re loading PDFs converted to text, they’ll often have headers, footers, or page numbers repeated every page — these don’t add meaning but hurt retrieval.

Academic articles might include references sections that aren’t useful for Q&A.

Some docs may contain weird line breaks, bullet symbols, or boilerplate disclaimers (like “Confidential — Company Use Only”).

Cleaning rules should strip these patterns so embeddings focus on meaningful content.

2) What chunk size/overlap tradeoff makes sense for your domain?

For short FAQs: use smaller chunks (200–400 chars) with little/no overlap — keeps answers tight.

For long PDFs or lecture notes: larger chunks (500–1,000 chars) with some overlap (50–150 chars) ensure context continuity across paragraphs.

Tradeoff: smaller chunks = better precision but risk losing context; larger chunks = better context but can dilute retrieval accuracy.